In [5]:
import pandas as pd
import hashlib


In [2]:
INTERNAL_EMAIL = "besar@ibbagency.com"

In [6]:
emails = pd.read_csv("../data/raw/emails.csv")

emails["sender_email"]    = emails["sender_email"].str.lower().str.strip()
emails["recipient_email"] = emails["recipient_email"].str.lower().str.strip()

emails = emails[emails['direction'] == 'Sent']
emails = emails[(emails['recipient_email'] != INTERNAL_EMAIL) & (emails['sender_email'] != INTERNAL_EMAIL)]

emails["date"] = pd.to_datetime(emails["date"], utc=True, errors="coerce")

emails.columns, emails.shape

(Index(['email_id', 'source_folder', 'folder', 'source_file', 'date',
        'direction', 'sender_name', 'sender_email', 'recipient_name',
        'recipient_email', 'cc', 'subject', 'body_raw', 'body_clean',
        'in_reply_to', 'references', 'is_reply', 'has_attachments'],
       dtype='str'),
 (236, 18))

In [4]:
emails[['direction', 'sender_email', 'recipient_email']].sample(5)

,direction,sender_email,recipient_email
440,Sent,shend@ibbagency.com,petcare@petplate.com
325,Sent,shend@ibbagency.com,contactstrangerstories@gmail.com
341,Sent,shend@ibbagency.com,ewucrew@protonmail.com
470,Sent,shend@ibbagency.com,info.mysterious7@gmail.com
331,Sent,shend@ibbagency.com,partnerships@helixsleep.com


In [ ]:
prospects = (
    emails.sort_values("date")
    .groupby("recipient_email", as_index=False)
    .agg(
        prospect_name=("recipient_name", "first"),
        first_contacted_date=("date", "min"),
        source_folder=("source_folder", "first"),
        first_subject=("subject", "first"),
    )
    .rename(columns={"recipient_email": "prospect_email"})
    .sort_values("first_contacted_date")
    .reset_index(drop=True)
)

In [8]:
prospects

,prospect_email,prospect_name,first_contacted_date,source_folder,first_subject
0,ft-20aeea-f4771965@ft.glockdb.com,NaN,2026-07-24 14:34:50+00:00,sent,Test E-Mail
1,elizabeaver@auth.glockdb.com,NaN,2026-07-24 16:40:04+00:00,sent,Test Email
2,martinawm@gemings.awsapps.com,NaN,2026-07-24 16:46:53+00:00,sent,Testing My Business Domain
3,stewartzjones22@gmail.com,NaN,2026-07-25 16:06:47+00:00,sent,Regarding Spam Testing Tools
4,llionelcohenbr@gmail.com,NaN,2026-07-27 01:14:02+00:00,sent,Glock Apps Email Tester
...,...,...,...,...,...
113,press@lockly.com,NaN,2026-11-04 22:57:43+00:00,sent,Lockly on YouTube
114,maria@hubbyesim.com,NaN,2026-11-06 08:27:40+00:00,sent,Creator partnerships for Hubby eSim
115,level8@level8cases.com,NaN,2026-11-06 08:41:19+00:00,sent,YouTube creators for Level8
116,support@hushed.com,NaN,2026-11-06 08:52:24+00:00,sent,Hushed on YouTube


In [9]:
prospects.insert(0, "prospect_id", [f"P{i:03d}" for i in range(1, len(prospects) + 1)])

In [11]:
len(prospects), prospects["prospect_email"].is_unique

(118, True)

In [ ]:
prospects.to_csv(
    "../data/processed/prospects.csv",
    index=False
)

In [23]:
prospects = pd.read_csv("../data/prospects.csv")

In [24]:
prospects

,prospect_id,prospect_email,prospect_name,first_contacted_date,source_folder,first_subject,is_brand
0,P013,thisismonsters@gmail.com,This Is Monsters,2026-08-10 16:06:35+00:00,sent,Sponsorship idea for this is Monsters,False
1,P014,ytthemissing@gmail.com,The Missing Enigma,2026-08-10 17:24:08+00:00,sent,Sponsorship Opportunity for The Missing Enigma,False
2,P015,business@criminallylisted.com,Criminally Listed,2026-08-12 16:20:10+00:00,sent,Sponsorship Opportunity For Criminally Listed,False
3,P016,dire.trip@gmail.com,Dire Trip,2026-08-12 23:33:21+00:00,sent,Sponsorship Opportunity for Dire Trip YouTube ...,False
4,P017,contact@thesneakystudios.com,Sneaky Sushii,2026-08-17 15:41:24+00:00,sent,Sponsorship Opportunity for Sneaky Shushii,False
...,...,...,...,...,...,...,...
100,P114,press@lockly.com,Lockly,2026-11-04 22:57:43+00:00,sent,Lockly on YouTube,True
101,P115,maria@hubbyesim.com,Hubby eSIM,2026-11-06 08:27:40+00:00,sent,Creator partnerships for Hubby eSim,True
102,P116,level8@level8cases.com,Level8,2026-11-06 08:41:19+00:00,sent,YouTube creators for Level8,True
103,P117,support@hushed.com,Hushed,2026-11-06 08:52:24+00:00,sent,Hushed on YouTube,True


In [25]:
prospects.is_brand.value_counts()

is_brand
False    59
True     46
Name: count, dtype: int64

In [26]:
creators = prospects[
    prospects["is_brand"] == False
].copy()

brands = prospects[
    prospects["is_brand"] == True
].copy()

In [27]:
creators.shape

(59, 7)

In [28]:
brands.shape

(46, 7)

In [29]:
creators.to_csv(
    "../data/processed/creators.csv",
    index=False
)

brands.to_csv(
    "../data/processed/brands.csv",
    index=False
)